# VM.AI — Notebook trainer
**Colab notebook** — requires in /content/drive/MyDrive/VM.AI a clone of the most up-to-date repo.

---

## 1. Mount Google Drive

In [31]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Clone or update the repo

In [32]:
import os
from google.colab import userdata

# Store your token in Colab Secrets (🔑 icon on the left sidebar)
# Key name: GITHUB_TOKEN
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Infiteri/VM.AI.git"
REPO_DIR = "/content/drive/MyDrive/VM.AI"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest changes...")
    %cd {REPO_DIR}
    !git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print(f"Working directory: {os.getcwd()}")

Repo already exists — pulling latest changes...
/content/drive/MyDrive/VM.AI
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
Working directory: /content/drive/MyDrive/VM.AI


In [33]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 3. Install dependencies

In [ ]:
!pip install -q transformers datasets huggingface_hub pyyaml numpy torch


shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spi

## 4. Verify GPU


In [35]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — training will be very slow on CPU.")


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## 5. Paths & environment config


In [ ]:
import sys, os

REPO_DIR = "/content/drive/MyDrive/VM.AI"
SRC_DIR  = os.path.join(REPO_DIR, "src")

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, "parser"))

os.chdir(os.path.join(SRC_DIR, "parser"))
print("cwd:", os.getcwd())
print("sys.path entries added:", SRC_DIR)


cwd: /content/drive/MyDrive/VM.AI/src/parser
sys.path entries added: /content/drive/MyDrive/VM.AI/src


In [ ]:
import torch

class EnvConfig:
    def __init__(self, env: str):
        self.env = env
        if env == "local":
            self.setup_local()
        elif env == "colab":
            self.setup_colab()
        else:
            raise ValueError(f"Unknown env '{env}'")

        if not torch.cuda.is_available():
            self.fp16                   = False
            self.dataloader_num_workers = 0
            self.dataloader_pin_memory  = False

    def setup_local(self):
        self.max_limit                   = 100
        self.num_train_epochs            = 3
        self.per_device_train_batch_size = 8
        self.per_device_eval_batch_size  = 8
        self.gradient_accumulation_steps = 16
        self.fp16                        = True
        self.dataloader_num_workers      = 4
        self.dataloader_pin_memory       = True
        self.logging_steps               = 10

        BASE = "."
        self.model_cache = f"{BASE}/models/google-t5/t5-small"
        self.output_dir  = f"{BASE}/models/finetuned_parser"
        self.data_path   = f"{BASE}/data"         

    def setup_colab(self):
        self.max_limit                   = 100
        self.num_train_epochs            = 3
        self.per_device_train_batch_size = 8
        self.per_device_eval_batch_size  = 8
        self.gradient_accumulation_steps = 16
        self.fp16                        = True
        self.dataloader_num_workers      = 2
        self.dataloader_pin_memory       = False  
        self.logging_steps               = 10

        BASE = "/content/drive/MyDrive/VM.AI"
        self.model_cache = f"{BASE}/models/google-t5/t5-small"
        self.output_dir  = f"{BASE}/models/finetuned_parser"
        self.data_path   = f"{BASE}/data"         

cfg = EnvConfig("colab")
print(f"Env           : {cfg.env}")
print(f"Max limit     : {cfg.max_limit}")
print(f"Epochs        : {cfg.num_train_epochs}")
print(f"fp16          : {cfg.fp16}")
print(f"Model cache   : {cfg.model_cache}")
print(f"Output dir    : {cfg.output_dir}")
print(f"Data path     : {cfg.data_path}")


Env           : colab
Max limit     : 100
Epochs        : 3
fp16          : True
Model cache   : /content/drive/MyDrive/VM.AI/models/google-t5/t5-small
Output dir    : /content/drive/MyDrive/VM.AI/models/finetuned_parser
Data path     : /content/drive/MyDrive/VM.AI/data


## 6. Download base model (T5-small)

In [38]:
import os
from huggingface_hub import snapshot_download

os.makedirs(cfg.model_cache, exist_ok=True)

if not os.path.exists(cfg.model_cache) or not os.listdir(cfg.model_cache):
    print("Downloading google/t5-small to Drive...")
    snapshot_download(repo_id="google-t5/t5-small", local_dir=cfg.model_cache)
else:
    print(f"Model already cached at {cfg.model_cache}")


Model already cached at /content/drive/MyDrive/VM.AI/models/google-t5/t5-small


## 7. Load data and generate synthetic dataset

In [39]:
import vars
from yaml_parser import VMAI_YamlParser
from data_generator import VMAI_DataGenerator

data_file = os.path.join(cfg.data_path, vars.SYNTHETIC_DATASET_PATH)
print(f"Loading data from: {data_file}")

parser = VMAI_YamlParser(data_file)
parser.load_yaml()
training_data = parser.parse()

dataset = VMAI_DataGenerator(training_data).generate(cfg.max_limit)
split   = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]
test_dataset  = split["test"]

print(f"Train examples : {len(train_dataset)}")
print(f"Test examples  : {len(test_dataset)}")


Loading data from: /content/drive/MyDrive/VM.AI/data/VMAI_SYNTHETIC_Data.yaml
Train examples : 90
Test examples  : 10


## 8. Tokenize

In [40]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_cache)

def tokenize_function(examples):
    inputs = tokenizer(
        examples["input_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    targets = tokenizer(
        examples["target_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    labels = targets["input_ids"]
    labels = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels
    ]
    inputs["labels"] = np.array(labels, dtype=np.int64)
    return inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function,  batched=True)

cols = ["input_ids", "attention_mask", "labels"]
tokenized_train.set_format(type="torch", columns=cols)
tokenized_test.set_format( type="torch", columns=cols)

print("Tokenization complete.")


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenization complete.


## 9. Load model (resume if checkpoint exists)

In [41]:
import torch
from transformers import T5ForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(cfg.output_dir, exist_ok=True)

if os.path.exists(cfg.output_dir) and os.listdir(cfg.output_dir):
    print("Resuming from checkpoint...")
    model = T5ForConditionalGeneration.from_pretrained(cfg.output_dir)
else:
    print("Loading base T5-small...")
    model = T5ForConditionalGeneration.from_pretrained(cfg.model_cache)

model.to(device)
print(f"Model on: {device}")


Resuming from checkpoint...


Loading weights:   0%|          | 0/131 [00:03<?, ?it/s]

Model on: cuda


## 10. Train

In [42]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

training_args = Seq2SeqTrainingArguments(
    output_dir=                  cfg.output_dir,
    eval_strategy=               "epoch",
    save_strategy=               "epoch",
    learning_rate=               2e-5,
    weight_decay=                0.01,
    save_total_limit=            2,
    predict_with_generate=       True,
    push_to_hub=                 False,
    remove_unused_columns=       False,
    optim=                       "adafactor",
    num_train_epochs=            cfg.num_train_epochs,
    per_device_train_batch_size= cfg.per_device_train_batch_size,
    per_device_eval_batch_size=  cfg.per_device_eval_batch_size,
    gradient_accumulation_steps= cfg.gradient_accumulation_steps,
    fp16=                        cfg.fp16,
    dataloader_num_workers=      cfg.dataloader_num_workers,
    dataloader_pin_memory=       cfg.dataloader_pin_memory,
    logging_steps=               cfg.logging_steps,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=         model,
    args=          training_args,
    train_dataset= tokenized_train,
    eval_dataset=  tokenized_test,
    data_collator= data_collator,
)

print("Starting training...")
trainer.train()


Starting training...


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,No log,0.054977
2,No log,0.054363
3,No log,0.054363


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please conside

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please conside

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=1.0355301698048909, metrics={'train_runtime': 11.794, 'train_samples_per_second': 22.893, 'train_steps_per_second': 0.254, 'total_flos': 9135571599360.0, 'train_loss': 1.0355301698048909, 'epoch': 3.0})

## 11. Save model to Drive

In [43]:
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"✅ Model saved to {cfg.output_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/drive/MyDrive/VM.AI/models/finetuned_parser


## 12. Quick sanity test


In [44]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

model_test = T5ForConditionalGeneration.from_pretrained(cfg.output_dir).to(device)
tok_test   = AutoTokenizer.from_pretrained(cfg.output_dir)

test_input = "remind me to call John tomorrow at 3pm"
inputs = tok_test(test_input, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model_test.generate(**inputs, max_length=64)

result = tok_test.decode(output_ids[0], skip_special_tokens=True)
print(f"Input  : {test_input}")
print(f"Output : {result}")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Input  : remind me to call John tomorrow at 3pm
Output : TASK: call John | MATE | MATE | MATE | MATE | MATE | TIME: 3pm
